# Modular SAXS/WAXS Analysis Pipeline
This notebook runs the analysis using modular functions and settings from `sample_settings.csv`.

In [2]:
import sys
import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# Add root directory to python path for imports
sys.path.append('..')

from classes_and_functions.read_data import DataReader, read_background_directory, read_background_file
from classes_and_functions.electrochemical_data_filtration import process_electrochemical_data
from classes_and_functions.data_correction import DataCorrection
from classes_and_functions.utils import contourplot

## 1. Load Settings and Select Sample

In [3]:
settings_df = pd.read_csv('sample_settings.csv')
sample_selector = widgets.Dropdown(
    options=settings_df['Sample_ID'].tolist(),
    description='Sample:',
)
display(sample_selector)

Dropdown(description='Sample:', options=('Without_iodine_20251017', 'HDF5_Test_Sample'), value='Without_iodine…

In [4]:
# Get settings for selected sample
sample_settings = settings_df[settings_df['Sample_ID'] == sample_selector.value].iloc[0]
print(f"Selected Sample: {sample_settings['Sample_ID']}")
print(f"Data Format: {sample_settings['Data_Format']}")

Selected Sample: Without_iodine_20251017
Data Format: Individual


## 2. Read Data

In [5]:
reader = DataReader(
    data_format=sample_settings['Data_Format'],
    data_path=sample_settings['Data_Path'],
    mpr_file_path=sample_settings.get('MPR_File_Path') if pd.notna(sample_settings.get('MPR_File_Path')) else None
)
all_data = reader.read_data()
print(f"Loaded {len(all_data['SAXS'])} SAXS files and {len(all_data['WAXS'])} WAXS files.")

Loaded 571 SAXS files and 571 WAXS files.


## 3. Data Correction (Background & Capillary)

In [6]:
# Process SAXS Background
if pd.notna(sample_settings.get('Empty_Dir_SAXS')):
    empty_saxs_dict = read_background_directory(sample_settings['Empty_Dir_SAXS'], '*_0_*.dat')
    capillary_saxs = read_background_file(sample_settings['Capillary_SAXS']) if pd.notna(sample_settings.get('Capillary_SAXS')) else None
    print('Correcting SAXS data...')
    corrector = DataCorrection(all_data['SAXS'])
    all_data['SAXS'] = corrector.subtract_background(empty_cell_dict=empty_saxs_dict, capillary_df=capillary_saxs, start_row=1, end_row_idx=843)

# Process WAXS Background
if pd.notna(sample_settings.get('Empty_Dir_WAXS')):
    empty_waxs_dict = read_background_directory(sample_settings['Empty_Dir_WAXS'], '*_1_*.dat')
    capillary_waxs = read_background_file(sample_settings['Capillary_WAXS']) if pd.notna(sample_settings.get('Capillary_WAXS')) else None
    print('Correcting WAXS data...')
    corrector = DataCorrection(all_data['WAXS'])
    all_data['WAXS'] = corrector.subtract_background(empty_cell_dict=empty_waxs_dict, capillary_df=capillary_waxs, start_row=1)

FileNotFoundError: [Errno 2] No such file or directory: 'e:\\Github_repositories\\SAXS_WAXS\\Notebooks_for_SWAXS\\iodine_project\\host_data\\capillary_20251017\\capillary_20251017_1_0001.dat'

## 4. Process Electrochemical Data

In [ ]:
ec_results = process_electrochemical_data(all_data, rate=sample_settings['Discharge_Rate'])

## 5. Generate SAXS Contour Plot

In [ ]:
contourplot(
    plot_type='saxs',
    data_normalization='absolute',
    data_dictionary=all_data['SAXS'],
    new_time_dict=all_data['TimeStamps_SAXS'],
    min_limit=sample_settings['SAXS_Min_Limit'],
    max_limit=sample_settings['SAXS_Max_Limit'],
    y_axis_mode='time',
    elec_df_2=ec_results['filtered_dataframe'] if ec_results else None,
    applied_current=ec_results['applied_current'] if ec_results else None,
    active_material_mass=ec_results['active_material_mass'] if ec_results else None
)

## 6. Generate WAXS Contour Plot

In [ ]:
contourplot(
    plot_type='waxs',
    data_normalization='absolute',
    data_dictionary=all_data['WAXS'],
    new_time_dict=all_data['TimeStamps_WAXS'],
    min_limit=sample_settings['WAXS_Min_Limit'],
    max_limit=sample_settings['WAXS_Max_Limit'],
    y_axis_mode='time',
    elec_df_2=ec_results['filtered_dataframe'] if ec_results else None,
    applied_current=ec_results['applied_current'] if ec_results else None,
    active_material_mass=ec_results['active_material_mass'] if ec_results else None
)